# FastText vs. Word2Vec: evidence report

Trains FastText and plain Word2Vec (cbow + sg) at a handful of (dim, window)
points, sharing a single corpus load and vocabulary count across every model
(see `compare_lib.py`'s `run_comparison()`), and compares them three ways:
1. **timing** -- wall-clock seconds per config
2. **RSA** -- do FastText and Word2Vec agree on which words are similar to
   which (representational similarity analysis over a shared word sample),
   independent of the two spaces' unrelated, arbitrarily-rotated axes
3. **predictive power** -- scored through `03_evaluation/evaluation.py`'s
   own real pipeline (replication norms, extended norms, frequency counts),
   not a bespoke metric

This exists because FastText's subword n-grams (`min_n=3, max_n=6`, matched
to subs2vec's Table 1 -- see `05_manuscript/manuscript.Rmd:314`) make it
~1.4-1.9x slower to train than Word2Vec (measured on `af`), which compounds
badly across 58 remaining languages x 60 configs. `af` alone isn't a fair
test of whether that cost is worth it, though -- Afrikaans is unusually
morphologically *simple* for its family, which is exactly the case where
FastText's subword generalization should matter least. `README.md`'s
coverage set spans agglutinative, fusional, isolating, and templatic
morphology, and Latin, Cyrillic, Devanagari, Thai, and Hangul scripts, to
see whether the tradeoff looks different for languages where subwords
should actually help.

**Two ways to run this notebook:**
- **Overnight, the whole coverage set at once** (section 1 below) --
  `cl.run_comparison_batch()` builds each language's corpus if needed,
  trains and scores everything, keeps going if one language fails, and
  writes `REPORT.md` at the end. This is the one to kick off before bed.
- **One language at a time** (section 2) -- for debugging or a quick look
  at a single language before committing to an overnight run. Requires
  `corpora/corpus-{language}.txt` to already exist (steps 1-5 of
  `run_language_pipeline.ipynb`).

In [1]:
import os
import sys

HERE = os.path.dirname(os.path.abspath(__file__)) if "__file__" in dir() else os.getcwd()
REPO_ROOT = os.path.abspath(os.path.join(HERE, "..", ".."))
sys.path.insert(0, HERE)
import compare_lib as cl

cl.basedir = REPO_ROOT

version = "2018"    # '2018' or '2024'

# A diagonal sample across the full (dim, window) grid rather than all 30
# combinations -- five points span dim=50..500 and window=1..6 while
# keeping the per-language cost small. Edit freely; more points = more
# confidence, at roughly linear extra cost. Used by both sections below.
configs = [(50, 1), (100, 2), (200, 3), (300, 4), (500, 6)]

## 1. Overnight: run the whole coverage set
in:  nothing extra needed -- builds each language's corpus itself if it
     doesn't exist yet (download, clean, prune, concatenate; skipped for
     languages already prepared, same as `run_language_pipeline.ipynb`)
out: `models/{language}/*_wxd.csv.bz2` per language (gitignored -- see
     README.md), `results/{language}_{version}_{timing,rsa,predictive}.csv`,
     `results/batch_summary.csv`, and `REPORT.md`

Runs `cl.CORE_LANGUAGES` by default (`af`, `eu`, `kk`, `ta`, `hi`, `mk`,
`th`, `vi`, `ko`). Add `cl.EXTENDED_LANGUAGES` (`he`, `ar`, `zh`) too if you
have the time budget for it -- see README.md's coverage table for why
they're pricier. One language failing (network hiccup, missing corpus
dependency, disk space) doesn't stop the rest; check `results/batch_summary.csv`
or the printed summary at the end for anything that needs a rerun.

In [2]:
languages = ['zh', "th"] # cl.CORE_LANGUAGES + cl.EXTENDED_LANGUAGES  # or cl.CORE_LANGUAGES + cl.EXTENDED_LANGUAGES for full coverage
summary = cl.run_comparison_batch(languages, version=version, configs=configs, overwrite = False)


[1/2] zh
Loading zh corpus + vocab (shared across every config below)...
34470844 sentences, 6607309 unique tokens, workers=15
  [1/20] zhft_50_1_cbow: already exists, skipping training (recorded 2050.3s)
  [2/20] zhwv_50_1_cbow: already exists, skipping training (recorded 1449.6s)
  [3/20] zhft_50_1_sg: already exists, skipping training (recorded 2203.9s)
  [4/20] zhwv_50_1_sg: already exists, skipping training (recorded 1768.0s)
  [5/20] zhft_100_2_cbow: already exists, skipping training (recorded 2236.6s)
  [6/20] zhwv_100_2_cbow: already exists, skipping training (recorded 1675.3s)
  zhft_100_2_sg: 1960.2s, 1131521 words
  [7/20] zhft_100_2_sg: trained
  zhwv_100_2_sg: 1594.7s, 1131521 words
  [8/20] zhwv_100_2_sg: trained
  zhft_200_3_cbow: 1515.1s, 1131521 words
  [9/20] zhft_200_3_cbow: trained
  zhwv_200_3_cbow: 1041.8s, 1131521 words
  [10/20] zhwv_200_3_cbow: trained
  zhft_200_3_sg: 3179.2s, 1131521 words
  [11/20] zhft_200_3_sg: trained
  zhwv_200_3_sg: 2456.1s, 1131521 wo

### View the generated report

In [3]:
from IPython.display import Markdown, display

report_path = os.path.join(REPO_ROOT, "experiments", "fasttext_vs_word2vec", "REPORT.md")
with open(report_path) as f:
    display(Markdown(f.read()))

# FastText vs. Word2Vec -- evidence report

Auto-generated by `compare_lib.generate_report()`. Covers 12 language(s): `af`, `ar`, `eu`, `he`, `hi`, `kk`, `ko`, `mk`, `ta`, `th`, `vi`, `zh`.

See `README.md` for methodology and the full coverage plan.

## Run summary

| language | status | error |
|---|---|---|
| zh | ok |  |
| th | ok |  |

## Overall: timing

Mean Word2Vec speedup over FastText, by language (>1 = word2vec faster):

| language | word2vec_speedup |
|---|---|
| af | 2.260 |
| ar | 1.580 |
| eu | 2.440 |
| he | 1.890 |
| hi | 2.180 |
| kk | 2.520 |
| ko | 1.610 |
| mk | 2.230 |
| ta | 2.660 |
| th | 2.040 |
| vi | 1.720 |
| zh | 1.360 |

**Overall mean speedup across all languages run so far: 2.04x**

## Overall: vector geometry (RSA)

Pearson r between FastText's and Word2Vec's word-similarity structure, vs. each family's own cbow-vs-sg agreement (a same-family reference point -- if fasttext_vs_word2vec is *lower* than either cbow_vs_sg column for a language, that's a language where the algorithm choice reshapes geometry more than the training objective does, unlike what af showed).

| language | fasttext | fasttext_vs_word2vec | word2vec |
|---|---|---|---|
| af | 0.638 | 0.784 | 0.807 |
| eu | 0.694 | 0.786 | 0.787 |
| he | 0.568 | 0.814 | 0.626 |
| hi | 0.669 | 0.856 | 0.814 |
| kk | 0.627 | 0.735 | 0.733 |
| ko | 0.744 | 0.900 | 0.779 |
| mk | 0.598 | 0.777 | 0.750 |
| ta | 0.708 | 0.660 | 0.757 |
| th | 0.475 | 0.847 | 0.719 |
| vi | 0.693 | 0.828 | 0.777 |
| zh | 0.776 | 0.959 | 0.800 |

## Overall: predictive power

Mean r on normalized vectors, by language x eval_type x family:

| language | eval_type | fasttext | word2vec |
|---|---|---|---|
| af | counts | 0.156 | 0.164 |
| af | norms | 0.346 | 0.294 |
| eu | counts | 0.092 | 0.107 |
| he | counts | 0.077 | 0.077 |
| he | norms | 0.000 | 0.000 |
| hi | counts | 0.148 | 0.182 |
| kk | counts | 0.177 | 0.202 |
| ko | counts | 0.109 | 0.127 |
| mk | counts | 0.073 | 0.094 |
| ta | counts | 0.126 | 0.174 |
| th | counts | 0.175 | 0.186 |
| th | norms | 0.731 | 0.716 |
| vi | counts | 0.209 | 0.223 |
| zh | counts | 0.051 | 0.057 |
| zh | norms | 0.547 | 0.546 |

## Per-language detail

### `af`

**Timing (seconds):**

| dim | window | alg | fasttext | word2vec |
|---|---|---|---|---|
| 50 | 1 | cbow | 82.900 | 45.700 |
| 50 | 1 | sg | 89.100 | 55.000 |
| 100 | 2 | cbow | 86.900 | 45.700 |
| 100 | 2 | sg | 112.000 | 65.300 |
| 200 | 3 | cbow | 136.300 | 75.300 |
| 200 | 3 | sg | 232.900 | 107.900 |
| 300 | 4 | cbow | 208.300 | 67.700 |
| 300 | 4 | sg | 329.800 | 190.000 |
| 500 | 6 | cbow | 395.000 | 85.000 |
| 500 | 6 | sg | 658.200 | 307.300 |

**RSA (Pearson r):**

| dim | window | alg | fasttext | fasttext_vs_word2vec | word2vec |
|---|---|---|---|---|---|
| 50 | 1 | cbow |  | 0.824 |  |
| 50 | 1 | cbow_vs_sg | 0.718 |  | 0.853 |
| 50 | 1 | sg |  | 0.884 |  |
| 100 | 2 | cbow |  | 0.779 |  |
| 100 | 2 | cbow_vs_sg | 0.658 |  | 0.838 |
| 100 | 2 | sg |  | 0.836 |  |
| 200 | 3 | cbow |  | 0.755 |  |
| 200 | 3 | cbow_vs_sg | 0.627 |  | 0.807 |
| 200 | 3 | sg |  | 0.791 |  |
| 300 | 4 | cbow |  | 0.746 |  |
| 300 | 4 | cbow_vs_sg | 0.609 |  | 0.788 |
| 300 | 4 | sg |  | 0.756 |  |
| 500 | 6 | cbow |  | 0.743 |  |
| 500 | 6 | cbow_vs_sg | 0.577 |  | 0.750 |
| 500 | 6 | sg |  | 0.727 |  |

**Predictive power (normalized vectors, mean r by eval_type x family):**

| eval_type | fasttext | word2vec |
|---|---|---|
| counts | 0.156 | 0.164 |
| norms | 0.346 | 0.294 |

### `ar`

**Timing (seconds):**

| dim | window | alg | fasttext | word2vec |
|---|---|---|---|---|
| 50 | 1 | cbow | 4710.900 | 3835.000 |
| 50 | 1 | sg | 4467.200 | 3431.500 |
| 100 | 2 | cbow | 4801.900 | 3852.800 |
| 100 | 2 | sg | 5818.500 | 4031.100 |
| 200 | 3 | cbow | 5463.400 | 3173.700 |
| 200 | 3 | sg | 10075.000 | 6127.000 |
| 300 | 4 | cbow | 8874.600 | 3515.000 |
| 300 | 4 | sg | 14999.500 | 9646.500 |
| 500 | 6 | cbow | 14151.000 |  |

**Predictive power:** no local replication norms, extended norms, or frequency counts for this language yet -- see `03_evaluation/evaluation.py`'s `load_replication_norms`/`load_extended_norms`/`load_count_freqs`. Timing and RSA above are still valid; nothing to score predictive power against until that data exists (e.g. run `eval_inputs/build_counts_tokenized.py` for frequency counts).

### `eu`

**Timing (seconds):**

| dim | window | alg | fasttext | word2vec |
|---|---|---|---|---|
| 50 | 1 | cbow | 100.800 | 56.900 |
| 50 | 1 | sg | 114.900 | 66.500 |
| 100 | 2 | cbow | 149.500 | 73.300 |
| 100 | 2 | sg | 189.200 | 109.400 |
| 200 | 3 | cbow | 223.400 | 87.100 |
| 200 | 3 | sg | 305.300 | 169.800 |
| 300 | 4 | cbow | 330.200 | 103.800 |
| 300 | 4 | sg | 518.700 | 242.100 |
| 500 | 6 | cbow | 617.600 | 116.500 |
| 500 | 6 | sg | 957.200 | 452.900 |

**RSA (Pearson r):**

| dim | window | alg | fasttext | fasttext_vs_word2vec | word2vec |
|---|---|---|---|---|---|
| 50 | 1 | cbow |  | 0.809 |  |
| 50 | 1 | cbow_vs_sg | 0.746 |  | 0.843 |
| 50 | 1 | sg |  | 0.869 |  |
| 100 | 2 | cbow |  | 0.722 |  |
| 100 | 2 | cbow_vs_sg | 0.663 |  | 0.805 |
| 100 | 2 | sg |  | 0.866 |  |
| 200 | 3 | cbow |  | 0.707 |  |
| 200 | 3 | cbow_vs_sg | 0.676 |  | 0.782 |
| 200 | 3 | sg |  | 0.848 |  |
| 300 | 4 | cbow |  | 0.708 |  |
| 300 | 4 | cbow_vs_sg | 0.689 |  | 0.766 |
| 300 | 4 | sg |  | 0.832 |  |
| 500 | 6 | cbow |  | 0.707 |  |
| 500 | 6 | cbow_vs_sg | 0.694 |  | 0.738 |
| 500 | 6 | sg |  | 0.792 |  |

**Predictive power (normalized vectors, mean r by eval_type x family):**

| eval_type | fasttext | word2vec |
|---|---|---|
| counts | 0.092 | 0.107 |

### `he`

**Timing (seconds):**

| dim | window | alg | fasttext | word2vec |
|---|---|---|---|---|
| 50 | 1 | cbow | 1334.500 | 819.100 |
| 50 | 1 | sg | 1455.300 | 960.000 |
| 100 | 2 | cbow | 1665.100 | 924.800 |
| 100 | 2 | sg | 1954.700 | 1627.600 |
| 200 | 3 | cbow | 2511.500 | 1191.600 |
| 200 | 3 | sg | 4026.600 | 2186.700 |
| 300 | 4 | cbow | 2769.900 | 1280.400 |
| 300 | 4 | sg | 5232.000 | 3217.600 |
| 500 | 6 | cbow | 4715.800 | 1370.500 |
| 500 | 6 | sg | 9029.000 | 5621.600 |

**RSA (Pearson r):**

| dim | window | alg | fasttext | fasttext_vs_word2vec | word2vec |
|---|---|---|---|---|---|
| 50 | 1 | cbow |  | 0.831 |  |
| 50 | 1 | cbow_vs_sg | 0.591 |  | 0.767 |
| 50 | 1 | sg |  | 0.839 |  |
| 100 | 2 | cbow |  | 0.862 |  |
| 100 | 2 | cbow_vs_sg | 0.569 |  | 0.677 |
| 100 | 2 | sg |  | 0.821 |  |
| 200 | 3 | cbow |  | 0.846 |  |
| 200 | 3 | cbow_vs_sg | 0.567 |  | 0.635 |
| 200 | 3 | sg |  | 0.823 |  |
| 300 | 4 | cbow |  | 0.827 |  |
| 300 | 4 | cbow_vs_sg | 0.564 |  | 0.605 |
| 300 | 4 | sg |  | 0.827 |  |
| 500 | 6 | cbow |  | 0.632 |  |
| 500 | 6 | cbow_vs_sg | 0.548 |  | 0.445 |
| 500 | 6 | sg |  | 0.830 |  |

**Predictive power (normalized vectors, mean r by eval_type x family):**

| eval_type | fasttext | word2vec |
|---|---|---|
| counts | 0.077 | 0.077 |
| norms | 0.000 | 0.000 |

### `hi`

**Timing (seconds):**

| dim | window | alg | fasttext | word2vec |
|---|---|---|---|---|
| 50 | 1 | cbow | 61.900 | 35.400 |
| 50 | 1 | sg | 68.400 | 44.400 |
| 100 | 2 | cbow | 82.800 | 43.000 |
| 100 | 2 | sg | 125.400 | 83.800 |
| 200 | 3 | cbow | 166.200 | 60.900 |
| 200 | 3 | sg | 237.800 | 144.000 |
| 300 | 4 | cbow | 216.300 | 74.000 |
| 300 | 4 | sg | 402.900 | 254.200 |
| 500 | 6 | cbow | 433.700 | 96.400 |
| 500 | 6 | sg | 807.500 | 464.800 |

**RSA (Pearson r):**

| dim | window | alg | fasttext | fasttext_vs_word2vec | word2vec |
|---|---|---|---|---|---|
| 50 | 1 | cbow |  | 0.858 |  |
| 50 | 1 | cbow_vs_sg | 0.786 |  | 0.889 |
| 50 | 1 | sg |  | 0.924 |  |
| 100 | 2 | cbow |  | 0.828 |  |
| 100 | 2 | cbow_vs_sg | 0.721 |  | 0.854 |
| 100 | 2 | sg |  | 0.921 |  |
| 200 | 3 | cbow |  | 0.811 |  |
| 200 | 3 | cbow_vs_sg | 0.660 |  | 0.816 |
| 200 | 3 | sg |  | 0.896 |  |
| 300 | 4 | cbow |  | 0.812 |  |
| 300 | 4 | cbow_vs_sg | 0.620 |  | 0.783 |
| 300 | 4 | sg |  | 0.868 |  |
| 500 | 6 | cbow |  | 0.820 |  |
| 500 | 6 | cbow_vs_sg | 0.558 |  | 0.731 |
| 500 | 6 | sg |  | 0.820 |  |

**Predictive power (normalized vectors, mean r by eval_type x family):**

| eval_type | fasttext | word2vec |
|---|---|---|
| counts | 0.148 | 0.182 |

### `kk`

**Timing (seconds):**

| dim | window | alg | fasttext | word2vec |
|---|---|---|---|---|
| 50 | 1 | cbow | 80.200 | 43.500 |
| 50 | 1 | sg | 89.400 | 49.100 |
| 100 | 2 | cbow | 113.900 | 54.300 |
| 100 | 2 | sg | 149.200 | 84.100 |
| 200 | 3 | cbow | 210.100 | 74.400 |
| 200 | 3 | sg | 270.900 | 144.000 |
| 300 | 4 | cbow | 281.800 | 86.000 |
| 300 | 4 | sg | 457.100 | 223.400 |
| 500 | 6 | cbow | 553.700 | 100.800 |
| 500 | 6 | sg | 883.700 | 421.400 |

**RSA (Pearson r):**

| dim | window | alg | fasttext | fasttext_vs_word2vec | word2vec |
|---|---|---|---|---|---|
| 50 | 1 | cbow |  | 0.721 |  |
| 50 | 1 | cbow_vs_sg | 0.668 |  | 0.804 |
| 50 | 1 | sg |  | 0.820 |  |
| 100 | 2 | cbow |  | 0.666 |  |
| 100 | 2 | cbow_vs_sg | 0.604 |  | 0.761 |
| 100 | 2 | sg |  | 0.828 |  |
| 200 | 3 | cbow |  | 0.661 |  |
| 200 | 3 | cbow_vs_sg | 0.609 |  | 0.723 |
| 200 | 3 | sg |  | 0.803 |  |
| 300 | 4 | cbow |  | 0.670 |  |
| 300 | 4 | cbow_vs_sg | 0.625 |  | 0.704 |
| 300 | 4 | sg |  | 0.771 |  |
| 500 | 6 | cbow |  | 0.685 |  |
| 500 | 6 | cbow_vs_sg | 0.628 |  | 0.673 |
| 500 | 6 | sg |  | 0.724 |  |

**Predictive power (normalized vectors, mean r by eval_type x family):**

| eval_type | fasttext | word2vec |
|---|---|---|
| counts | 0.177 | 0.202 |

### `ko`

**Timing (seconds):**

| dim | window | alg | fasttext | word2vec |
|---|---|---|---|---|
| 50 | 1 | cbow | 319.900 | 198.800 |
| 50 | 1 | sg | 340.700 | 241.200 |
| 100 | 2 | cbow | 375.200 | 234.500 |
| 100 | 2 | sg | 584.200 | 392.700 |
| 200 | 3 | cbow | 447.400 | 260.000 |
| 200 | 3 | sg | 929.700 | 661.700 |
| 300 | 4 | cbow | 575.400 | 293.800 |
| 300 | 4 | sg | 1296.000 | 1087.600 |
| 500 | 6 | cbow | 908.200 | 428.400 |
| 500 | 6 | sg | 2562.500 | 1597.600 |

**RSA (Pearson r):**

| dim | window | alg | fasttext | fasttext_vs_word2vec | word2vec |
|---|---|---|---|---|---|
| 50 | 1 | cbow |  | 0.831 |  |
| 50 | 1 | cbow_vs_sg | 0.798 |  | 0.825 |
| 50 | 1 | sg |  | 0.888 |  |
| 100 | 2 | cbow |  | 0.881 |  |
| 100 | 2 | cbow_vs_sg | 0.755 |  | 0.800 |
| 100 | 2 | sg |  | 0.914 |  |
| 200 | 3 | cbow |  | 0.895 |  |
| 200 | 3 | cbow_vs_sg | 0.733 |  | 0.768 |
| 200 | 3 | sg |  | 0.929 |  |
| 300 | 4 | cbow |  | 0.893 |  |
| 300 | 4 | cbow_vs_sg | 0.720 |  | 0.757 |
| 300 | 4 | sg |  | 0.938 |  |
| 500 | 6 | cbow |  | 0.887 |  |
| 500 | 6 | cbow_vs_sg | 0.712 |  | 0.743 |
| 500 | 6 | sg |  | 0.943 |  |

**Predictive power (normalized vectors, mean r by eval_type x family):**

| eval_type | fasttext | word2vec |
|---|---|---|
| counts | 0.109 | 0.127 |

### `mk`

**Timing (seconds):**

| dim | window | alg | fasttext | word2vec |
|---|---|---|---|---|
| 50 | 1 | cbow | 137.300 | 83.900 |
| 50 | 1 | sg | 150.500 | 90.700 |
| 100 | 2 | cbow | 208.600 | 120.500 |
| 100 | 2 | sg | 264.500 | 159.600 |
| 200 | 3 | cbow | 301.900 | 131.300 |
| 200 | 3 | sg | 430.300 | 226.300 |
| 300 | 4 | cbow | 455.200 | 147.800 |
| 300 | 4 | sg | 691.000 | 346.400 |
| 500 | 6 | cbow | 805.300 | 176.900 |
| 500 | 6 | sg | 1356.200 | 774.100 |

**RSA (Pearson r):**

| dim | window | alg | fasttext | fasttext_vs_word2vec | word2vec |
|---|---|---|---|---|---|
| 50 | 1 | cbow |  | 0.804 |  |
| 50 | 1 | cbow_vs_sg | 0.673 |  | 0.819 |
| 50 | 1 | sg |  | 0.790 |  |
| 100 | 2 | cbow |  | 0.762 |  |
| 100 | 2 | cbow_vs_sg | 0.619 |  | 0.778 |
| 100 | 2 | sg |  | 0.779 |  |
| 200 | 3 | cbow |  | 0.756 |  |
| 200 | 3 | cbow_vs_sg | 0.579 |  | 0.739 |
| 200 | 3 | sg |  | 0.784 |  |
| 300 | 4 | cbow |  | 0.753 |  |
| 300 | 4 | cbow_vs_sg | 0.568 |  | 0.720 |
| 300 | 4 | sg |  | 0.794 |  |
| 500 | 6 | cbow |  | 0.743 |  |
| 500 | 6 | cbow_vs_sg | 0.551 |  | 0.696 |
| 500 | 6 | sg |  | 0.804 |  |

**Predictive power (normalized vectors, mean r by eval_type x family):**

| eval_type | fasttext | word2vec |
|---|---|---|
| counts | 0.073 | 0.094 |

### `ta`

**Timing (seconds):**

| dim | window | alg | fasttext | word2vec |
|---|---|---|---|---|
| 50 | 1 | cbow | 75.200 | 38.300 |
| 50 | 1 | sg | 80.900 | 41.100 |
| 100 | 2 | cbow | 101.400 | 46.500 |
| 100 | 2 | sg | 133.200 | 69.700 |
| 200 | 3 | cbow | 178.600 | 69.500 |
| 200 | 3 | sg | 241.800 | 114.400 |
| 300 | 4 | cbow | 253.700 | 73.400 |
| 300 | 4 | sg | 389.200 | 190.700 |
| 500 | 6 | cbow | 504.400 | 83.600 |
| 500 | 6 | sg | 761.400 | 316.200 |

**RSA (Pearson r):**

| dim | window | alg | fasttext | fasttext_vs_word2vec | word2vec |
|---|---|---|---|---|---|
| 50 | 1 | cbow |  | 0.693 |  |
| 50 | 1 | cbow_vs_sg | 0.716 |  | 0.806 |
| 50 | 1 | sg |  | 0.805 |  |
| 100 | 2 | cbow |  | 0.615 |  |
| 100 | 2 | cbow_vs_sg | 0.707 |  | 0.775 |
| 100 | 2 | sg |  | 0.748 |  |
| 200 | 3 | cbow |  | 0.593 |  |
| 200 | 3 | cbow_vs_sg | 0.711 |  | 0.756 |
| 200 | 3 | sg |  | 0.685 |  |
| 300 | 4 | cbow |  | 0.597 |  |
| 300 | 4 | cbow_vs_sg | 0.714 |  | 0.739 |
| 300 | 4 | sg |  | 0.648 |  |
| 500 | 6 | cbow |  | 0.604 |  |
| 500 | 6 | cbow_vs_sg | 0.694 |  | 0.709 |
| 500 | 6 | sg |  | 0.613 |  |

**Predictive power (normalized vectors, mean r by eval_type x family):**

| eval_type | fasttext | word2vec |
|---|---|---|
| counts | 0.126 | 0.174 |

### `th`

**Timing (seconds):**

| dim | window | alg | fasttext | word2vec |
|---|---|---|---|---|
| 50 | 1 | cbow | 161.600 | 106.400 |
| 50 | 1 | sg | 190.500 | 131.400 |
| 100 | 2 | cbow | 264.600 | 148.700 |
| 100 | 2 | sg | 345.700 | 234.000 |
| 200 | 3 | cbow | 366.700 | 167.500 |
| 200 | 3 | sg | 580.000 | 322.700 |
| 300 | 4 | cbow | 543.700 | 195.700 |
| 300 | 4 | sg | 902.400 | 524.100 |
| 500 | 6 | cbow | 954.400 | 245.400 |
| 500 | 6 | sg | 1699.300 | 948.500 |

**RSA (Pearson r):**

| dim | window | alg | fasttext | fasttext_vs_word2vec | word2vec |
|---|---|---|---|---|---|
| 50 | 1 | cbow |  | 0.869 |  |
| 50 | 1 | cbow_vs_sg | 0.616 |  | 0.784 |
| 50 | 1 | sg |  | 0.872 |  |
| 100 | 2 | cbow |  | 0.843 |  |
| 100 | 2 | cbow_vs_sg | 0.468 |  | 0.732 |
| 100 | 2 | sg |  | 0.856 |  |
| 200 | 3 | cbow |  | 0.835 |  |
| 200 | 3 | cbow_vs_sg | 0.437 |  | 0.712 |
| 200 | 3 | sg |  | 0.848 |  |
| 300 | 4 | cbow |  | 0.826 |  |
| 300 | 4 | cbow_vs_sg | 0.431 |  | 0.701 |
| 300 | 4 | sg |  | 0.848 |  |
| 500 | 6 | cbow |  | 0.814 |  |
| 500 | 6 | cbow_vs_sg | 0.425 |  | 0.667 |
| 500 | 6 | sg |  | 0.855 |  |

**Predictive power (normalized vectors, mean r by eval_type x family):**

| eval_type | fasttext | word2vec |
|---|---|---|
| counts | 0.175 | 0.186 |
| norms | 0.731 | 0.716 |

### `vi`

**Timing (seconds):**

| dim | window | alg | fasttext | word2vec |
|---|---|---|---|---|
| 50 | 1 | cbow | 270.100 | 217.400 |
| 50 | 1 | sg | 362.900 | 267.000 |
| 100 | 2 | cbow | 396.800 | 255.600 |
| 100 | 2 | sg | 607.700 | 443.800 |
| 200 | 3 | cbow | 560.500 | 287.000 |
| 200 | 3 | sg | 1036.800 | 647.700 |
| 300 | 4 | cbow | 803.400 | 342.700 |
| 300 | 4 | sg | 1558.200 | 1042.800 |
| 500 | 6 | cbow | 1373.700 | 490.300 |
| 500 | 6 | sg | 2910.700 | 1924.600 |

**RSA (Pearson r):**

| dim | window | alg | fasttext | fasttext_vs_word2vec | word2vec |
|---|---|---|---|---|---|
| 50 | 1 | cbow |  | 0.858 |  |
| 50 | 1 | cbow_vs_sg | 0.718 |  | 0.800 |
| 50 | 1 | sg |  | 0.929 |  |
| 100 | 2 | cbow |  | 0.816 |  |
| 100 | 2 | cbow_vs_sg | 0.712 |  | 0.798 |
| 100 | 2 | sg |  | 0.879 |  |
| 200 | 3 | cbow |  | 0.803 |  |
| 200 | 3 | cbow_vs_sg | 0.702 |  | 0.790 |
| 200 | 3 | sg |  | 0.824 |  |
| 300 | 4 | cbow |  | 0.780 |  |
| 300 | 4 | cbow_vs_sg | 0.694 |  | 0.775 |
| 300 | 4 | sg |  | 0.802 |  |
| 500 | 6 | cbow |  | 0.796 |  |
| 500 | 6 | cbow_vs_sg | 0.639 |  | 0.723 |
| 500 | 6 | sg |  | 0.790 |  |

**Predictive power (normalized vectors, mean r by eval_type x family):**

| eval_type | fasttext | word2vec |
|---|---|---|
| counts | 0.209 | 0.223 |

### `zh`

**Timing (seconds):**

| dim | window | alg | fasttext | word2vec |
|---|---|---|---|---|
| 50 | 1 | cbow | 2050.300 | 1449.600 |
| 50 | 1 | sg | 2203.900 | 1768.000 |
| 100 | 2 | cbow | 2236.600 | 1675.300 |
| 100 | 2 | sg | 1960.200 | 1594.700 |
| 200 | 3 | cbow | 1515.100 | 1041.800 |
| 200 | 3 | sg | 3179.200 | 2456.100 |
| 300 | 4 | cbow | 1875.400 | 1255.800 |
| 300 | 4 | sg | 4911.700 | 3868.500 |
| 500 | 6 | cbow | 2717.900 | 1630.100 |
| 500 | 6 | sg | 8803.700 | 7280.100 |

**RSA (Pearson r):**

| dim | window | alg | fasttext | fasttext_vs_word2vec | word2vec |
|---|---|---|---|---|---|
| 50 | 1 | cbow |  | 0.911 |  |
| 50 | 1 | cbow_vs_sg | 0.803 |  | 0.864 |
| 50 | 1 | sg |  | 0.963 |  |
| 100 | 2 | cbow |  | 0.945 |  |
| 100 | 2 | cbow_vs_sg | 0.776 |  | 0.805 |
| 100 | 2 | sg |  | 0.968 |  |
| 200 | 3 | cbow |  | 0.958 |  |
| 200 | 3 | cbow_vs_sg | 0.768 |  | 0.782 |
| 200 | 3 | sg |  | 0.971 |  |
| 300 | 4 | cbow |  | 0.961 |  |
| 300 | 4 | cbow_vs_sg | 0.767 |  | 0.776 |
| 300 | 4 | sg |  | 0.974 |  |
| 500 | 6 | cbow |  | 0.961 |  |
| 500 | 6 | cbow_vs_sg | 0.764 |  | 0.771 |
| 500 | 6 | sg |  | 0.979 |  |

**Predictive power (normalized vectors, mean r by eval_type x family):**

| eval_type | fasttext | word2vec |
|---|---|---|
| counts | 0.051 | 0.057 |
| norms | 0.547 | 0.546 |


## 2. One language at a time (manual / debugging)
Skip this section for the normal overnight workflow -- it's for testing a
single language interactively before committing it to a batch run, or for
rerunning just one language by hand.

Set `language` below first. Requires `corpora/corpus-{language}.txt` to
already exist.
in: `corpora/corpus-{language}.txt` (built by `run_language_pipeline.ipynb` steps 1-5)

In [ ]:
language = "eu"     # two-letter code -- see README.md's coverage table

corpus_key = language if version == "2018" else f"{language}-{version}"
corpus_path = os.path.join(REPO_ROOT, "corpora", f"corpus-{corpus_key}.txt")
if not os.path.exists(corpus_path):
    raise FileNotFoundError(
        f"{corpus_path} doesn't exist yet -- run steps 1-5 of run_language_pipeline.ipynb "
        f"for language='{language}' first (through cp.concatenate_corpus(), nothing further needed)."
    )
print(f"Found {corpus_path} ({os.path.getsize(corpus_path) / 1e6:.1f}MB) -- ready.")

### 2.1 Run the comparison for this one language
in:  the corpus confirmed above
out: `experiments/fasttext_vs_word2vec/models/{language}/*_wxd.csv.bz2`
     (fasttext + word2vec, cbow + sg, at every config point -- kept fully
     separate from the real `models/` directory)
     `experiments/fasttext_vs_word2vec/results/{language}_{version}_{timing,rsa,predictive}.csv`

This is the slow step -- roughly `len(configs) * 4` model trainings
(FastText + Word2Vec, cbow + sg), each comparable in cost to one config of
the real pipeline. Five config points is a fraction of the real 60-config
sweep's time for this language.

In [ ]:
result = cl.run_comparison(language, version=version, configs=configs)

### 2.2 Look at this language's results
Same three angles the overnight report aggregates: timing, RSA
(vector-geometry agreement), and predictive power.

In [ ]:
import pandas as pd

print("--- timing: mean seconds by family/alg, and word2vec speedup ---")
timing = result["timing"]
print(timing.groupby(["family", "alg"])["seconds"].mean().round(1))
pivot = timing.pivot_table(index=["dim", "window", "alg"], columns="family", values="seconds")
pivot["word2vec_speedup"] = pivot["fasttext"] / pivot["word2vec"]
print()
print(pivot.round(2))

In [ ]:
print("--- RSA: fasttext_vs_word2vec agreement vs. each family's own cbow-vs-sg agreement ---")
rsa = result["rsa"]
print(rsa.groupby(["comparison", "alg"])[["pearson_r", "spearman_r"]].mean().round(3))

In [ ]:
print("--- predictive power: mean r / r-squared by eval_type x family x alg (normalized vectors only) ---")
pred = result["predictive"]
normalized_only = pred[pred["normalized"] == True]
print(normalized_only.groupby(["eval_type", "family", "alg"])[["r", "r-squared"]].mean().round(4))

## 3. Aggregate across whatever's been run so far
Reads every `results/*_timing.csv` / `*_rsa.csv` / `*_predictive.csv` on
disk (from either section above) into three combined tables -- the same
data `REPORT.md` is generated from, if you want to work with it directly
instead of reading the rendered report.

In [ ]:
import glob

results_dir = os.path.join(REPO_ROOT, "experiments", "fasttext_vs_word2vec", "results")

def load_all(kind):
    paths = sorted(glob.glob(os.path.join(results_dir, f"*_{kind}.csv")))
    if not paths:
        return pd.DataFrame()
    return pd.concat([pd.read_csv(p) for p in paths], ignore_index=True)

all_timing = load_all("timing")
all_rsa = load_all("rsa")
all_predictive = load_all("predictive")

print(f"{all_timing['language'].nunique() if len(all_timing) else 0} language(s) so far: "
      f"{sorted(all_timing['language'].unique()) if len(all_timing) else []}")

if len(all_timing):
    print()
    print("--- word2vec speedup by language ---")
    p = all_timing.pivot_table(index=["language", "dim", "window", "alg"], columns="family", values="seconds")
    p["word2vec_speedup"] = p["fasttext"] / p["word2vec"]
    print(p.groupby("language")["word2vec_speedup"].mean().round(2))

if len(all_predictive):
    print()
    print("--- predictive power by language x family (normalized vectors, mean r across eval types) ---")
    norm_only = all_predictive[all_predictive["normalized"] == True]
    print(norm_only.groupby(["language", "family"])["r"].mean().round(4))